# Scalability

Writes the two runtime figures from the published run on NYC taxi trips, and tabulates their runtimes for reference.

In [9]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats
from figures import (
    COLORS,
    COLUMN_WIDTH_PT,
    LEGEND_SIZE,
    ORDER,
    PAPER_STYLE,
    TEXT_WIDTH_PT,
    measure_label,
    set_size,
)

from experiments.measures import TABLE_MEASURES, measure_dimension
from experiments.scalability import COLUMN_SWEEP_ROWS

RESULTS = Path("../measurements/scalability/nyc_taxi")


def read(sweep):
    return pd.concat(pd.read_parquet(p) for p in RESULTS.glob(f"{sweep}/**/results.parquet"))


def per_repetition(sweep, x):
    # Median over the columns a measure applies to, one value per repetition.
    ok = read(sweep).query("status == 'ok'")
    ok = ok.assign(seconds=ok.fit_seconds + ok.score_seconds)
    return ok.groupby([x, "measure", "repetition"]).seconds.median().reset_index()


def stats(repetitions, x):
    # Mean and std over the repetitions.
    return repetitions.groupby([x, "measure"]).seconds.agg(["mean", "std", "count"]).reset_index()


column_repetitions = per_repetition("columns", "n_columns")
rows, columns = stats(per_repetition("rows", "n_rows"), "n_rows"), stats(column_repetitions, "n_columns")
llm = read("llm")
assert set(rows["count"]) == {10}
assert column_repetitions.groupby("n_columns").repetition.nunique().eq(10).all()
assert llm.groupby("n_rows").repetition.nunique().eq(10).all()

# SemanticDataAccuracy: a line fitted per repetition, evaluated at the exact request counts.
requests = pd.Series(json.loads((RESULTS / "manifest.json").read_text())["requests"])
requests.index = requests.index.astype(int)
fits = llm.groupby("repetition")[["requests", "score_seconds"]].apply(
    lambda g: np.polyfit(g.requests, g.score_seconds, 1)
)
coefficients = np.stack(fits.to_list())  # one (slope, intercept) row per repetition
# 95% confidence intervals of the mean line and its coefficients, with the repetition as the independent unit.
t = scipy.stats.t.ppf(0.975, len(coefficients) - 1)


def band(x):
    """The mean of the per-repetition lines at `x`, and the half-width of its 95% confidence interval."""
    lines = np.array([np.polyval(c, x) for c in coefficients])
    return lines.mean(axis=0), t * lines.std(axis=0, ddof=1) / np.sqrt(len(lines))


predicted, half_width = band(requests.to_numpy())
predicted = pd.Series(predicted, index=requests.index)
(slope, intercept), (slope_ci, intercept_ci) = (
    coefficients.mean(axis=0),
    t * coefficients.std(axis=0, ddof=1) / np.sqrt(len(coefficients)),
)
r2 = scipy.stats.linregress(llm.requests, llm.score_seconds).rvalue ** 2  # over all measurements
hours_per_million = predicted[1_000_000] / 3600
print(
    f"slope = {slope * 1000:.1f} ± {slope_ci * 1000:.1f} ms per request; "
    f"intercept = {intercept:.1f} ± {intercept_ci:.1f} s; R² = {r2:.2f}\n"
    f"{hours_per_million:.1f} ± {half_width[requests.index.get_loc(1_000_000)] / 3600:.1f} h for 1M rows; "
    f"{predicted.iloc[-1]:.0f} ± {half_width[-1]:.0f} s ({predicted.iloc[-1] / 86400:.0f} ± "
    f"{half_width[-1] / 86400:.0f} days) for the whole year ({requests.iloc[-1]} requests)"
)

slope = 30.5 ± 4.6 ms per request; intercept = 1.7 ± 2.3 s; R² = 0.76
8.5 ± 1.3 h for 1M rows; 1254844 ± 190057 s (15 ± 2 days) for the whole year (41169716 requests)


In [10]:
DIMENSIONS = ["Accuracy", "Completeness", "Consistency", "Currentness"]
PANEL_OF = {"Timeliness": "Currentness"}
CYCLE = [COLORS[s] for s in ORDER if s not in ("clean", "ward_intake")]
# The DQMs the number of columns affects: every table-scoped DQM, and Semantic Consistency, the one column-scoped
# DQM that reads the other columns. Record Currentness is table-scoped but reads only the timestamp columns.
WIDE = [m.__name__ for m in TABLE_MEASURES if m.__name__ != "RecordCurrentness"] + ["SemanticConsistency"]
# Each repetition adds the columns in its own order; every DQM in WIDE applies to any subset.
assert set(columns[columns.measure.isin(WIDE)]["count"]) == {10}
# Five panels leave no room for full names; the dimension is abbreviated everywhere, as the panel title names it.
ABBREVIATIONS = {
    "Accuracy": "Acc.",
    "Inaccuracy": "Inacc.",
    "Completeness": "Compl.",
    "Consistency": "Cons.",
    "Inconsistency": "Incons.",
    "Currentness": "Curr.",
    "Distribution": "Dist.",
}

paper = rows.assign(dimension=rows.measure.map(lambda m: PANEL_OF.get(measure_dimension(m), measure_dimension(m))))
# One panel per dimension over the rows, then the column sweep, which has no title.
panels = [(paper[paper.dimension == d], "n_rows", d) for d in DIMENSIONS]
panels.append((columns[columns.measure.isin(WIDE)], "n_columns", None))


def short_label(measure):
    return " ".join(ABBREVIATIONS.get(word, word) for word in measure_label(measure).split())


def slowest_first(frame, x):
    # Slowest first, so the legend reads in the order the lines arrive at the right edge.
    return frame[frame[x] == frame[x].max()].sort_values("mean", ascending=False).measure.to_list()


# One colour per measure across the figure, distinct within each panel. The column panel mixes measures from two
# dimension panels, so it is coloured first; the dimension panels take theirs from what is left.
color = {}
for frame, x, _ in [panels[-1], *panels[:-1]]:
    free = [c for c in CYCLE if c not in {color[m] for m in frame.measure.unique() if m in color}]
    for measure in slowest_first(frame, x):
        if measure not in color:
            color[measure] = free.pop(0)


def draw(ax, key, frame, x, title):
    """The panel's measures on `ax`; its title and legend on `key`, a bare axes above it."""
    measures = slowest_first(frame, x)
    rng = np.random.default_rng(0)
    for i, measure in enumerate(measures):
        g = frame[frame.measure == measure].sort_values(x)
        if x == "n_columns":
            # Each repetition measures its own random columns, so the spread is which columns, not noise, and
            # ± SD would reach below zero on the log axis. The mean is a line and every repetition a dot,
            # shifted sideways per DQM and jittered within it, so the dots of neighbouring lines stay apart.
            ax.plot(g[x], g["mean"], color=color[measure], lw=1.0, label=short_label(measure))
            dots = column_repetitions[column_repetitions.measure == measure]
            shift = (i - (len(measures) - 1) / 2) * 0.9 + rng.uniform(-0.3, 0.3, len(dots))
            ax.scatter(dots[x] + shift, dots.seconds, s=4, color=color[measure], alpha=0.6, lw=0, zorder=3)
            continue
        ax.errorbar(
            g[x],
            g["mean"],
            yerr=g["std"],
            color=color[measure],
            lw=1.0,
            elinewidth=0.8,
            capsize=1.5,
            label=short_label(measure),
        )
    key.axis("off")
    if title:
        key.set_title(title)
    key.legend(*ax.get_legend_handles_labels(), loc="upper left", borderaxespad=0)


# One row of panels, too narrow to hold their legends. Each legend hangs from a flat axes above its panel, and
# the layout makes room for the tallest, so the titles line up.
with plt.rc_context(PAPER_STYLE):
    fig, (keys, axes) = plt.subplots(
        2,
        5,
        figsize=set_size(TEXT_WIDTH_PT, subplots=(1, 4), height_ratio=2.5),
        sharey="row",
        height_ratios=[0.01, 1],
        width_ratios=[1, 1, 1, 1, 0.7],
        layout="constrained",
    )
    fig.get_layout_engine().set(w_pad=1 / 72, h_pad=1 / 72, wspace=0.03, hspace=0)
    for ax, key, panel in zip(axes, keys, panels, strict=True):
        draw(ax, key, *panel)
    for ax in axes[:4]:
        ax.sharex(axes[0])
        ax.set(xscale="log", xlabel="Rows", xticks=[1e3, 1e5, 1e7])
    # The column sweep is a different experiment: its own tick labels set it apart from the row panels.
    axes[4].set(xlabel="Columns", xticks=[5, 15, 25])
    axes[4].tick_params(labelleft=True)
    axes[0].set(yscale="log", ylabel="Seconds per Measure")
    fig.align_xlabels(axes)
    fig.savefig("../results/2026-09-23-scalability.pdf")
    plt.close(fig)

In [11]:
measured = llm.groupby("n_rows").score_seconds.agg(["mean", "std"])
FIT = COLORS["typo"]
PM = r"\pm" if PAPER_STYLE["text.usetex"] else "±"

with plt.rc_context(PAPER_STYLE):
    fig, (left, right) = plt.subplots(
        1, 2, figsize=set_size(COLUMN_WIDTH_PT, subplots=(1, 2), height_ratio=1.6), layout="constrained"
    )

    # Left: the fitted range on linear axes, where the band is wide enough to see, down to zero for the intercept.
    x = np.linspace(0, 1000, 101)
    mean, half = band(x)
    left.fill_between(x, mean - half, mean + half, color=FIT, alpha=0.25, lw=0)
    left.plot(x, mean, color=FIT, lw=1.0)
    left.scatter(llm.requests, llm.score_seconds, s=3, color="black", lw=0, alpha=0.6)
    left.set(xlabel="Rows", ylabel="Seconds", xlim=(0, 1000), ylim=(0, None), xticks=[0, 300, 600, 900])
    left.text(
        0.04,
        0.96,
        rf"${slope * 1000:.1f} {PM} {slope_ci * 1000:.1f}$\,ms/request"
        "\n"
        rf"${intercept:.1f} {PM} {intercept_ci:.1f}$\,s intercept"
        "\n"
        rf"$R^2 = {r2:.2f}$",
        transform=left.transAxes,
        va="top",
        fontsize=LEGEND_SIZE,
    )

    # Right: the same line carried to the full dataset, marked at the two sizes the text quotes. On log axes the
    # band is a hairline, so the intervals are written out.
    right.fill_between(requests.index, predicted - half_width, predicted + half_width, color=FIT, alpha=0.25, lw=0)
    right.plot(requests.index, predicted, color=FIT, lw=1.0)
    right.scatter(measured.index, measured["mean"], s=4, color="black", lw=0, zorder=3)
    for n, unit, scale, fmt in [(1_000_000, "h", 3600, ".1f"), (requests.index[-1], "d", 86400, ".0f")]:
        i = requests.index.get_loc(n)
        right.plot(n, predicted[n], "o", ms=2.5, color=FIT)
        right.annotate(
            rf"${predicted[n] / scale:{fmt}} {PM} {half_width[i] / scale:{fmt}}$\,{unit}",
            (n, predicted[n]),
            xytext=(-4, 2),
            textcoords="offset points",
            ha="right",
            fontsize=LEGEND_SIZE,
        )
    right.set(xscale="log", yscale="log", xlabel="Rows", xticks=[1e3, 1e5, 1e7])
    fig.savefig("../results/2026-09-23-scalability-llm.pdf")
    plt.close(fig)

## Runtime table

A reference for proofreading the runtimes the text quotes; it is not in the paper. A cell is `fit` plus `score`,
as in the figure: the median over the columns a DQM applies to, then the mean over ten repetitions ± the standard
deviation, in ms below a second. The SD is rounded to one significant figure, two when it starts with a 1, and the
mean to the same decimal. The column sweep lists only the DQMs its panel shows; a dash is a DQM that found no column
to apply to.

In [12]:
def runtime(mean, sd):
    """`mean ± sd` with its unit, in ms below a second."""
    unit, scale = ("s", 1) if mean >= 1 else ("ms", 1e3)
    mean, sd = mean * scale, sd * scale
    # The SD to one significant figure, two when it starts with a 1, and the mean to the same decimal.
    exponent = int(np.floor(np.log10(sd)))
    decimals = max(0, (2 if sd / 10**exponent < 2 else 1) - 1 - exponent)
    return f"{mean:.{decimals}f} ± {sd:.{decimals}f} {unit}"


def runtime_table(frame, x, name):
    """One row per DQM, grouped by the figure's panels, and one column per size."""
    cells = frame.assign(
        dimension=frame.measure.map(lambda m: PANEL_OF.get(measure_dimension(m), measure_dimension(m))),
        dqm=frame.measure.map(measure_label),
        runtime=[runtime(m, s) for m, s in zip(frame["mean"], frame["std"], strict=True)],
    )
    table = cells.pivot(index=["dimension", "dqm"], columns=x, values="runtime").fillna("–")
    return table.rename(columns="{:,}".format).rename_axis(index=[None, None], columns=name)


display(runtime_table(rows, "n_rows", "Rows"))
runtime_table(columns[columns.measure.isin(WIDE)], "n_columns", f"Columns ({COLUMN_SWEEP_ROWS:,} rows)")

Rows                                                   1,000  \
Accuracy     Data Accuracy Range            0.708 ± 0.014 ms   
             Risk of Data Set Inaccuracy    0.889 ± 0.011 ms   
             Syntactic Data Accuracy          0.79 ± 0.02 ms   
Completeness Empty Records                    2.44 ± 0.06 ms   
             Feature Completeness           0.304 ± 0.003 ms   
             Record Completeness              2.51 ± 0.06 ms   
             Value Completeness               0.70 ± 0.06 ms   
             Value Occurrence Completeness  0.377 ± 0.014 ms   
Consistency  Data Format Consistency          2.46 ± 0.04 ms   
             Data Record Consistency          2.88 ± 0.16 ms   
             Data Value Distribution        0.768 ± 0.016 ms   
             Risk of Data Inconsistency       1.59 ± 0.10 ms   
             Semantic Consistency              16.4 ± 1.3 ms   
Currentness  Feature Currentness              1.40 ± 0.04 ms   
             Record Currentness               2.81 ± 0.03 ms   
             Timeliness of Data Items         1.35 ± 0.16 ms   
             Timeliness of Update             1.26 ± 0.14 ms   
             Update Frequency                 2.01 ± 0.13 ms   

Rows                                                   3,000  \
Accuracy     Data Accuracy Range            0.724 ± 0.010 ms   
             Risk of Data Set Inaccuracy    0.918 ± 0.013 ms   
             Syntactic Data Accuracy          0.87 ± 0.02 ms   
Completeness Empty Records                    2.60 ± 0.07 ms   
             Feature Completeness           0.309 ± 0.004 ms   
             Record Completeness              2.69 ± 0.09 ms   
             Value Completeness               0.74 ± 0.12 ms   
             Value Occurrence Completeness    0.49 ± 0.03 ms   
Consistency  Data Format Consistency          6.41 ± 0.03 ms   
             Data Record Consistency            3.5 ± 0.4 ms   
             Data Value Distribution        1.304 ± 0.020 ms   
             Risk of Data Inconsistency       1.72 ± 0.06 ms   
             Semantic Consistency              27.7 ± 1.0 ms   
Currentness  Feature Currentness              1.47 ± 0.11 ms   
             Record Currentness               2.87 ± 0.09 ms   
             Timeliness of Data Items           1.4 ± 0.2 ms   
             Timeliness of Update             1.30 ± 0.16 ms   
             Update Frequency                 2.49 ± 0.11 ms   

Rows                                                  10,000  \
Accuracy     Data Accuracy Range            0.753 ± 0.011 ms   
             Risk of Data Set Inaccuracy    1.013 ± 0.017 ms   
             Syntactic Data Accuracy          1.08 ± 0.06 ms   
Completeness Empty Records                    3.01 ± 0.15 ms   
             Feature Completeness           0.322 ± 0.003 ms   
             Record Completeness                3.3 ± 0.2 ms   
             Value Completeness               0.69 ± 0.06 ms   
             Value Occurrence Completeness    0.82 ± 0.06 ms   
Consistency  Data Format Consistency         20.31 ± 0.12 ms   
             Data Record Consistency            4.8 ± 0.3 ms   
             Data Value Distribution          3.04 ± 0.07 ms   
             Risk of Data Inconsistency       1.95 ± 0.13 ms   
             Semantic Consistency                  55 ± 2 ms   
Currentness  Feature Currentness              1.65 ± 0.19 ms   
             Record Currentness               2.98 ± 0.19 ms   
             Timeliness of Data Items         1.43 ± 0.11 ms   
             Timeliness of Update             1.29 ± 0.07 ms   
             Update Frequency                 3.56 ± 0.17 ms   

Rows                                                  30,000  \
Accuracy     Data Accuracy Range            0.808 ± 0.011 ms   
             Risk of Data Set Inaccuracy    1.227 ± 0.013 ms   
             Syntactic Data Accuracy          1.69 ± 0.07 ms   
Completeness Empty Records                      4.0 ± 0.5 ms   
             Feature Complete

Columns (10,000,000 rows)                          5                10  \
Completeness Empty Records                 31 ± 5 ms        75 ± 20 ms   
             Record Completeness           47 ± 6 ms        98 ± 33 ms   
             Value Completeness       0.25 ± 0.05 ms  0.339 ± 0.010 ms   
Consistency  Data Record Consistency     1.1 ± 0.3 s       2.0 ± 0.4 s   
             Semantic Consistency            6 ± 6 s          10 ± 7 s   

Columns (10,000,000 rows)                           15              20  \
Completeness Empty Records                 163 ± 68 ms     135 ± 12 ms   
             Record Completeness          199 ± 102 ms     166 ± 15 ms   
             Value Completeness       0.462 ± 0.020 ms  0.57 ± 0.02 ms   
Consistency  Data Record Consistency       3.0 ± 0.2 s     3.9 ± 1.0 s   
             Semantic Consistency             14 ± 5 s        21 ± 4 s   

Columns (10,000,000 rows)                         25  
Completeness Empty Records               180 ± 29 ms  
             Record Completeness         211 ± 47 ms  
             Value Completeness       0.70 ± 0.07 ms  
Consistency  Data Record Consistency     3.9 ± 0.6 s  
             Semantic Consistency       24.3 ± 0.5 s